# Top-k + selection-frequency barrier (redesigned fix)

The first Top-k + barrier attempt (`mixed_regularizer_experiment.py`) penalized raw pre-activation sign, which fights Top-k's hard mask directly (the barrier wants every one of 1024 features firing ~half the time; the mask discards all but 92 regardless) -- MSE exploded 6-32x and got WORSE with more regularization.

This version (`topk_selection_barrier_sae.py`) penalizes each feature's SELECTION FREQUENCY instead -- a smooth proxy for "was this feature one of the k chosen" -- leaving Top-k's per-input adaptive selection otherwise untouched, and drops the redundant budget term (Top-k already fixes sparsity exactly).

**A bug was caught and fixed during local smoke-testing**: the first version trained a plain SimpleSAE manually and returned it for evaluation, but SimpleSAE's own `.forward()` doesn't know about top-k selection -- `evaluate_full()` silently evaluated a fully-dense reconstruction the model was never trained to produce (MSE ~20-33, sparsity 0.000). Fixed by using the `TopKSAE` class directly, whose `.forward()` bakes the same selection in for both training and evaluation.

**After the fix, a 1-epoch smoke test shows**: sparsity now correctly matches Top-k's target (0.910, vs. the raw-barrier version's drift to 0.72-0.78) and purity/ablate both improve meaningfully (purity ~12-13x vanilla, ablate modestly above vanilla) -- but MSE is still elevated (13-20x vanilla) even at low lambda. Real multi-epoch numbers needed to know if that stabilizes or is a genuine remaining problem.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
!python topk_selection_barrier_sae.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambda-barrier 0.001 0.005 0.01 0.05

In [ ]:
!zip -r topk_selection_barrier_results.zip results/topk_selection_barrier
from google.colab import files
files.download('topk_selection_barrier_results.zip')